In [ ]:
import os
import pandas as pd
import numpy as np
import xgboost as xgb
import kagglehub
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

In [ ]:
dataset_path = kagglehub.dataset_download("brllrb/uber-and-lyft-dataset-boston-ma")
print("Path to dataset files:", dataset_path)

rides_file = os.path.join(dataset_path, 'rideshare_kaggle.csv')
df_merged = pd.read_csv(rides_file)

print(f"Dataset shape: {df_merged.shape}")

In [ ]:
missing_price_cabs = df_merged[df_merged['price'].isnull()]['cab_type'].unique()
print(f"Cab types with missing prices: {missing_price_cabs}")

initial_rows = len(df_merged)
df_merged = df_merged[df_merged['cab_type'] != 'Taxi']
print(f"Filtered out {initial_rows - len(df_merged)} standard Taxi rides (out of scope for ML pricing).")

df_merged.dropna(subset=['price'], inplace=True)

df_merged = df_merged[(df_merged['distance'] > 0) & (df_merged['price'] > 0)]


features = ['distance', 'cab_type', 'name', 'surge_multiplier', 'temperature', 'short_summary', 'hour', 'price']
df_final = df_merged[features].copy()

df_final['temperature'] = df_final['temperature'].fillna(df_final['temperature'].median())
df_final['short_summary'] = df_final['short_summary'].fillna('Unknown')

print(f"\nFinal cleaned dataset shape ready for XGBoost: {df_final.shape}")
df_final.head()

In [ ]:

X = df_final.drop(columns=['price'])
y = df_final['price']

# Optional: Un-comment the line below to test on a smaller subset if your laptop is struggling
# X, _, y, _ = train_test_split(X, y, train_size=100000, random_state=42) 

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.1, random_state=42)


numeric_features = ['distance', 'surge_multiplier', 'temperature']
categorical_features = ['cab_type', 'name', 'hour', 'short_summary']

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ])

In [ ]:
baseline_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', Ridge(alpha=1.0))
])

baseline_pipeline.fit(X_train, y_train)
y_pred_base = baseline_pipeline.predict(X_test)

base_rmse = mean_squared_error(y_test, y_pred_base)
base_mae = mean_absolute_error(y_test, y_pred_base)
base_r2 = r2_score(y_test, y_pred_base)

print(f"Baseline RMSE: ${base_rmse:.2f}")
print(f"Baseline MAE:  ${base_mae:.2f}")
print(f"Baseline R-Squared: {base_r2:.3f}")

In [ ]:
xgb_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', xgb.XGBRegressor(
        objective='reg:squarederror',
        n_estimators=100, 
        learning_rate=0.1,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        n_jobs=-1 
    ))
])

xgb_pipeline.fit(X_train, y_train)
y_pred_xgb = xgb_pipeline.predict(X_test)

xgb_rmse = mean_squared_error(y_test, y_pred_xgb)
xgb_mae = mean_absolute_error(y_test, y_pred_xgb)
xgb_r2 = r2_score(y_test, y_pred_xgb)

print(f"XGBoost RMSE: ${xgb_rmse:.2f}")
print(f"XGBoost MAE:  ${xgb_mae:.2f}")
print(f"XGBoost R-Squared: {xgb_r2:.3f}")

In [ ]:
improvement_pct = ((base_rmse - xgb_rmse) / base_rmse) * 100

print(f"The XGBoost architecture reduced fare prediction error (RMSE) by {improvement_pct:.1f}% compared to the linear baseline.")
print(f"Average dollar variance reduced from ${base_rmse:.2f} to ${xgb_rmse:.2f} per ride.")


In [ ]:
from sklearn.ensemble import RandomForestRegressor, StackingRegressor

print("--- Training Advanced Stacking Ensemble ---")

# 1. Define the base models (The "Traders")
# We use both Bagging (Random Forest) and Boosting (XGBoost)
base_estimators = [
    ('rf', RandomForestRegressor(
        n_estimators=100,
        max_depth=6,
        random_state=42,
        n_jobs=-1
    )),
    ('xgb', xgb.XGBRegressor(
        objective='reg:squarederror',
        n_estimators=100,
        learning_rate=0.1,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        n_jobs=-1
    ))
]

# 2. Define the Meta-Learner (The "Portfolio Manager")
# Ridge handles multicollinearity well, making it perfect for deciding between two strong tree models.
meta_learner = Ridge(alpha=1.0)

# 3. Build the Stacking Regressor
stacking_model = StackingRegressor(
    estimators=base_estimators,
    final_estimator=meta_learner,
    cv=3, # 3-fold cross-validation to train the meta-learner
    n_jobs=-1
)

# 4. Integrate into the Pipeline
stacking_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', stacking_model)
])

# 5. Train and Evaluate
stacking_pipeline.fit(X_train, y_train)
y_pred_stack = stacking_pipeline.predict(X_test)

stack_rmse = mean_squared_error(y_test, y_pred_stack)
stack_mae = mean_absolute_error(y_test, y_pred_stack)
stack_r2 = r2_score(y_test, y_pred_stack)

print(f"Stacking Ensemble RMSE: ${stack_rmse:.2f}")
print(f"Stacking Ensemble MAE:  ${stack_mae:.2f}")
print(f"Stacking Ensemble R-Squared: {stack_r2:.3f}")

# Calculate the ultimate improvement
final_improvement = ((base_rmse - stack_rmse) / base_rmse) * 100
print(f"\nFinal Edge: Stacking reduced overall error by {final_improvement:.1f}% from the baseline.")